 # Workflow for a transformation pathway of a single node energy system with perfect foresight

 In this application of the ETHOS.FINE framework, a transformation pathway of a energy system is modeled and optimized.

 All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

 The workflow is structures as follows:
 1. Required packages are imported and the input data path is set
 2. An energy system model instance is created
 3. Commodity sources are added to the energy system model
 4. Commodity conversion components are added to the energy system model
 5. Commodity storages are added to the energy system model
 6. Commodity sinks are added to the energy system model
 7. Material sinks are added to the energy system model
 8. Material sources are addeed to the energy system model
 9. Material conversions are added to the energy system model for recycling processes
 10. The energy system model is optimized
 11. Selected optimization results are presented


 # 1. Import required packages and set input data path

 The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd


cwd = Path.cwd()
data = getData()

 # 2. Create an energy system model instance

 The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

 The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {"GermanyRegion"}
commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "hydrogen": r"GW$_{H_{2},LHV}$",
    "evs": r"GWh",
}
commodities = {"electricity", "hydrogen", "evs"}
materials = {
    "lithium",
    "cobalt",
    "EVBatteries_lithium_scrap",
    "EVBatteries_cobalt_scrap",
}
materialUnitsDict = {
    "lithium": "tons/h",
    "cobalt": "tons/h",
    "EVBatteries_lithium_scrap": "tons/h",
    "EVBatteries_cobalt_scrap": "tons/h",
}

numberOfTimeSteps = 8760
hoursPerTimeStep = 1

In [3]:
initial_material_cost = {
    "lithium": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
    "cobalt": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
}

 # 2.1 define Transformation Pathway parameters

 Transformation Pathway Analyses can be run by setting a number of investment periods
 larger than 1, which is the default value and results in a single year optimization.

In [4]:
numberOfInvestmentPeriods = 3
startYear = 2020
interval = 5

In [5]:
pathwayBalanceLimit = pd.DataFrame(
    columns=["GermanyRegion", "Total", "lowerBound"],
    index=["Lithium_Resources", "Cobalt_Resources"],
)
pathwayBalanceLimit.loc["Lithium_Resources"] = [None, 14100, False]
pathwayBalanceLimit.loc["Cobalt_Resources"] = [None, 14100, False]

In [6]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    startYear=startYear,
    investmentPeriodInterval=interval,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    pathwayBalanceLimit=pathwayBalanceLimit,
    initialMaterialCost=initial_material_cost,
)

 # 3. Add commodity sources to the energy system model

 ## 3.1. Electricity sources

 ### Wind onshore

 change weather conditions for the different investment periods

In [7]:
operationRateMax = {}
operationRateMax[2020] = 1.2 * data["Wind (onshore), operationRateMax"]
operationRateMax[2025] = 0.7 * data["Wind (onshore), operationRateMax"]
operationRateMax[2030] = 1 * data["Wind (onshore), operationRateMax"]

 define existing stock for wind onshore turbines

In [8]:
stockWindonshoreCommissioning = {
    2015: 10,
}

stockWindoffshoreCommissioning = {
    2015: 15,
}

 define invest and opex per capacity for wind onshore turbines

In [9]:
investPerCapacityWind = {2015: 1.25, 2020: 1.1, 2025: 1, 2030: 0.95}

opexPerCapacityWind = {
    2015: 1.25 * 0.02,
    2020: 1.1 * 0.02,
    2025: 1 * 0.02,
    2030: 0.95 * 0.02,
}

 add wind onshore source to esM

In [10]:
esM.add(
    fn.Source(
        esM=esM,
        name="windonshore",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=3.1,
        opexPerCapacity=3.1 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        # stockCommissioning=stockWindonshoreCommissioning,
    )
)

 Full load hours:

In [11]:
data["Wind (onshore), operationRateMax"].sum()

2300.4069071646272

In [12]:
esM.add(
    fn.Source(
        esM=esM,
        name="EVBatteries",
        commodity="evs",
        hasCapacityVariable=True,
        investPerCapacity={2020: 110, 2025: 100, 2030: 90},
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=5,
        commissioningFix={2020: 230, 2025: 230, 2030: 230},
        materialIntensity={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 2.0}),
                "cobalt": pd.Series({"GermanyRegion": 2.0}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 1.9}),
                "cobalt": pd.Series({"GermanyRegion": 1.9}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 1.8}),
                "cobalt": pd.Series({"GermanyRegion": 1.8}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 0.7}),
                "cobalt": pd.Series({"GermanyRegion": 0.7}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 0.8}),
                "cobalt": pd.Series({"GermanyRegion": 0.8}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 0.9}),
                "cobalt": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

 # 4. Add conversion components to the energy system model

 ### Electrolyzers

 add component with constant invest and opex per capacity

In [13]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electroylzers",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=5,
    )
)

 # 5. Add commodity storages to the energy system model

 ## 5.1. Electricity storage

 ### Lithium ion batteries

 The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [14]:
esM.add(
    fn.Storage(
        esM=esM,
        name="batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        investPerCapacity={2020: 110, 2025: 100, 2030: 90},
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 4.0}),
                "cobalt": pd.Series({"GermanyRegion": 4.0}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 3.9}),
                "cobalt": pd.Series({"GermanyRegion": 3.9}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 3.8}),
                "cobalt": pd.Series({"GermanyRegion": 3.8}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 0.7}),
                "cobalt": pd.Series({"GermanyRegion": 0.7}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 0.8}),
                "cobalt": pd.Series({"GermanyRegion": 0.8}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 0.9}),
                "cobalt": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

In [15]:
esM.add(
    fn.Storage(
        esM=esM,
        name="SLbatteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        investPerCapacity={2020: 100, 2025: 900, 2030: 80},
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity={
            2020: {  # IP für 2020
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 5.0}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 5.0}),
            },
            2025: {  # IP für 2025
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 4.9}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 4.9}),
            },
            2030: {  # IP für 2030
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 4.8}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 4.8}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 0.7}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.7}),
            },
            2025: {  # IP für 2025
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 0.8}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.8}),
            },
            2030: {  # IP für 2030
                "EVBatteries_lithium_scrap": pd.Series({"GermanyRegion": 0.9}),
                "EVBatteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

 ## 5.2. Hydrogen storage

 ### Hydrogen filled salt caverns
 The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [16]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity={2020: 0.00011, 2025: 0.00009, 2030: 0.00009},
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

 # 6. Add commodity sinks to the energy system model

 ## 6.1. Electricity sinks

 ### Electricity demand

 vary the demand with the years - increasing demand by 30% per year

In [17]:
electricityDemand = {}
electricityDemand[2020] = (1 + 0 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2025] = (1 + 1 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2030] = (1 + 2 * 0.3) * data["Electricity demand, operationRateFix"]

esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=electricityDemand,
    )
)

In [18]:
target_annual_demand = {2020: 230, 2025: 230, 2030: 230}

EVDemand = {}
for year, annual_gwh in target_annual_demand.items():
    hourly_value = annual_gwh
    EVDemand[year] = pd.Series([hourly_value] * 8760)


esM.add(
    fn.Sink(
        esM=esM,
        name="EV demand",
        commodity="evs",
        hasCapacityVariable=False,
        operationRateFix=EVDemand,
    )
)

 ## 6.2. Hydrogen sinks

 ### Fuel cell electric vehicle (FCEV) demand

In [19]:
FCEV_penetration = 0.5

# vary the demand with the years - increasing demand by 25% per year
hydrogendDemand = {}
hydrogendDemand[2020] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)
hydrogendDemand[2025] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)
hydrogendDemand[2030] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)


esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=hydrogendDemand,
    )
)

# 7. Add material sinks to the energy system model

In [20]:
esM.generationMaterialSinks()

Existing material sinks: set()
Missing materials sinks: {'lithium', 'EVBatteries_cobalt_scrap', 'cobalt', 'EVBatteries_lithium_scrap'}
New sink added: Lithium demand
New sink added: Evbatteries_cobalt_scrap demand
New sink added: Cobalt demand
New sink added: Evbatteries_lithium_scrap demand


# 8. Add material sources to the energy system model


## 8.1 Add primary material sources with pathway balance limit


In [21]:
esM.add(
    fn.Source(
        esM=esM,
        name="Lithium supply",
        hasCapacityVariable=True,
        commodity="lithium",
        pathwayBalanceLimitID="Lithium_Resources",
        investPerCapacity={2020: 100, 2025: 100, 2030: 100},
        opexPerCapacity=1,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Cobalt supply",
        hasCapacityVariable=True,
        commodity="cobalt",
        pathwayBalanceLimitID="Cobalt_Resources",
        investPerCapacity={2020: 100, 2025: 100, 2030: 100},
        opexPerCapacity=1,
    )
)

In [22]:
esM.generationSecondaryMaterialSources()

Scrap commodities : {'EVBatteries_lithium_scrap', 'batteries_lithium_scrap', 'batteries_cobalt_scrap', 'SLbatteries_EVBatteries_cobalt_scrap_scrap', 'SLbatteries_EVBatteries_lithium_scrap_scrap', 'EVBatteries_cobalt_scrap'}
Existing secondary material sources: set()
Missing secondary material sources: {'batteries_cobalt_scrap', 'EVBatteries_lithium_scrap', 'SLbatteries_EVBatteries_cobalt_scrap_scrap', 'SLbatteries_EVBatteries_lithium_scrap_scrap', 'batteries_lithium_scrap', 'EVBatteries_cobalt_scrap'}
New scrap source added: batteries_cobalt_scrap
New scrap source added: EVBatteries_lithium_scrap
New scrap source added: SLbatteries_EVBatteries_cobalt_scrap_scrap
New scrap source added: SLbatteries_EVBatteries_lithium_scrap_scrap
New scrap source added: batteries_lithium_scrap
New scrap source added: EVBatteries_cobalt_scrap
Existing secondary material sinks: {'EVBatteries_lithium_scrap', 'EVBatteries_cobalt_scrap'}
Missing secondary material sinks: {'batteries_lithium_scrap', 'SLbatter

In [23]:
esM.add(
    fn.Sink(
        esM=esM,
        name="EVBatteries_lithium_scrap rec",
        hasCapacityVariable=False,
        commodity="EVBatteries_lithium_scrap",
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="EVBatteries_cobalt_scrap rec",
        hasCapacityVariable=False,
        commodity="EVBatteries_cobalt_scrap",
    )
)

# 9. Add recycling plants

In [24]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler batteries",
        physicalUnit=r"tons/h",
        commodityConversionFactors={
            "batteries_lithium_scrap": -1,
            "lithium": 1,
            "batteries_cobalt_scrap": -1,
            "cobalt": 1,
        },
        hasCapacityVariable=True,
        economicLifetime=33,
        # investPerCapacity = {2020: 1, 2025: 1.5, 2030: 2},
        # opexPerCapacity=0.002,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler EV batteries",
        physicalUnit=r"tons/h",
        commodityConversionFactors={
            "EVBatteries_lithium_scrap": -1,
            "lithium": 1,
            "EVBatteries_cobalt_scrap": -1,
            "cobalt": 1,
        },
        hasCapacityVariable=True,
        economicLifetime=33,
        # investPerCapacity = {2020: 1, 2025: 1.5, 2030: 2},
        # opexPerCapacity=0.002,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler batteries SL",
        physicalUnit=r"tons/h",
        commodityConversionFactors={
            "SLbatteries_EVBatteries_cobalt_scrap_scrap": -1,
            "lithium": 1,
            "SLbatteries_EVBatteries_lithium_scrap_scrap": -1,
            "cobalt": 1,
        },
        hasCapacityVariable=True,
        economicLifetime=33,
        # investPerCapacity = {2020: 1, 2025: 1.5, 2030: 2},
        # opexPerCapacity=0.002,
    )
)

 # 10. Optimize energy system model

 All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

In [25]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...


		(15.9357 sec)



In [26]:
esM.optimize(timeSeriesAggregation=True, solver="gurobi")

Time series aggregation specifications:
Number of typical periods:30, number of time steps per period:24, number of segments per period:12

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.2962 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.6734 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(2.5292 sec)

		(0.0001 sec)

Declaring shared potential constraint...
		(0.0006 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.5345 sec)

Declaring material demand constraints...
RHS_demand 2.0*commis_srcSnk[GermanyRegion,EVBatteries,0] + 4.0*commis_stor[GermanyRegion,batteries,0]
RHS_demand 1.9*commis_srcSnk[GermanyRegion,EVBatteries,1] + 3.9*commis_stor[German

/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component batteries
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component batteries
  warnings.warn(


for StorageModel ...  (3.4819sec)
		(9.4302 sec)



 # 11. Selected results output

In [27]:
total_cost = 0.0

for loc, mat in esM.pyM.initialMaterialSet:
    quantity = esM.pyM.initialMaterialSupply[loc, mat].value
    unit_cost = esM.initialMaterialCost[mat][loc]
    material_cost = quantity * unit_cost

    print(f"{mat}:")
    print(f"  Initial supply [tons]: {quantity:.6f}")
    print(f"  Unit cost: {unit_cost:.8f} [USD/tons]")
    print(f"  Cost contribution : {material_cost:.6f} [USD]")

    total_cost += material_cost

print(f"\nTotal cost initial supply: {total_cost:.6f} [USD]")

lithium:
  Initial supply [tons]: 862.880294
  Unit cost: 0.10000000 [USD/tons]
  Cost contribution : 86.288029 [USD]
cobalt:
  Initial supply [tons]: 862.880294
  Unit cost: 0.10000000 [USD/tons]
  Cost contribution : 86.288029 [USD]

Total cost initial supply: 172.576059 [USD]


In [28]:
import pyomo.environ as pyomo

initial_cost = sum(
    esM.pyM.initialMaterialSupply[loc, mat] * esM.initialMaterialCost[mat][loc]
    for loc, mat in esM.pyM.initialMaterialSet
)

print("Initial material cost:", pyomo.value(initial_cost))
print("Total Objective:", pyomo.value(esM.pyM.Obj))

Initial material cost: 172.57605879873392
Total Objective: 111805.31794928995


In [29]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

 ### Sources and Sink

 Show optimization summary

In [30]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of SourceSinkModel for year {year}")
    print(esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=year))


 Results of SourceSinkModel for year 2020
                                                          GermanyRegion
Component          Property        Unit                                
Cobalt demand      operation       [tons/h*h/a]              304.919781
                                   [tons/h*h]                304.919781
Cobalt supply      NPVcontribution [1e9 Euro]                   2.38699
                   TAC             [1e9 Euro/a]                0.553553
                   capacity        [tons/h]                    0.034808
                   capexCap        [1e9 Euro/a]                0.518745
                   commissioning   [tons/h]                    0.034808
                   invest          [1e9 Euro]                  3.480819
                   operation       [tons/h*h/a]              304.919781
                                   [tons/h*h]                304.919781
                   opexCap         [1e9 Euro/a]                0.034808
EV demand          op

 ### Conversion

 Show optimization summary

In [31]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of ConversionMpdel for year {year}")
    print(esM.getOptimizationSummary("ConversionModel", outputLevel=2, ip=year))


 Results of ConversionMpdel for year 2020
                                              GermanyRegion
Component     Property        Unit                         
Electroylzers NPVcontribution [1e9 Euro]           0.684107
              TAC             [1e9 Euro/a]         0.158647
              capacity        [GW$_{el}$]          1.151886
              capexCap        [1e9 Euro/a]         0.144249
              commissioning   [GW$_{el}$]          1.151886
              invest          [1e9 Euro]           0.575943
              operation       [GW$_{el}$*h/a]   6807.249824
                              [GW$_{el}$*h]     6807.249824
              opexCap         [1e9 Euro/a]         0.014399

 Results of ConversionMpdel for year 2025
                                                   GermanyRegion
Component          Property        Unit                         
Electroylzers      NPVcontribution [1e9 Euro]           0.626034
                   TAC             [1e9 Euro/a]         0.2

 ### Storage

 Show optimization summary

In [32]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of StorageModel for year {year}")
    print(esM.getOptimizationSummary("StorageModel", outputLevel=2, ip=year))


 Results of StorageModel for year 2020
                                                                  GermanyRegion
Component               Property           Unit                                
Salt caverns (hydrogen) NPVcontribution    [1e9 Euro]                  1.488908
                        TAC                [1e9 Euro/a]                0.345284
                        capacity           [GW$_{H_{2},LHV}$*h]      595.552401
                        capexCap           [1e9 Euro/a]                0.005819
                        commissioning      [GW$_{H_{2},LHV}$*h]      595.552401
                        invest             [1e9 Euro]                  0.065511
                        operationCharge    [GW$_{H_{2},LHV}$*h/a]   1833.207679
                                           [GW$_{H_{2},LHV}$*h]     1833.207679
                        operationDischarge [GW$_{H_{2},LHV}$*h/a]   1833.207679
                                           [GW$_{H_{2},LHV}$*h]     1833.207679
